In [ ]:
# TODO! we need:
# - figure out the best raw dataset split:
#   - for the FULL dataset? (need to figure out how long it would take to process it in full?)
#   - maybe do a full split reservation when processing the whole thing, and process subsets only?
#   - should the split be a stratified one? if so, based on what tags/metadata?
# - figure out what the training data preparation strategy should be:
#   - what code hints to use? in which proportions? and with what hyperparameters? and generated by what LLMs?
#   - are existing augmentations satisfactory (i.e. high-quality?)
#   - should we be providing obfuscated augmentations directly? and also of augmented prompts?
# - figure out what the evaluation data preparation strategy should be:
#   - what code issues to use? in which proportions? and with what hyperparameters? and generated by what LLMs?
#   - should we also build misleading documentation as a way to reverse-throw-off the models? (with false hints?)
#   - how many evaluation scenarios (problems) should we be targeting? and with how many inputs/outputs?
# ...

In [ ]:
import yaml

import pyine.data.traces.dataset_reader
import pyine.data.traces.dataset_utils
import pyine.data.utils.splits

dataset_paths = pyine.data.traces.dataset_utils.get_matching_dataset_paths(
    source_dataset_name="TACO",
    pattern="ref-10s10t.*of000026.*.lmdb",
)
split = pyine.data.utils.splits.get_dataset_split_result("TACO")
part_file_paths = pyine.data.utils.splits.get_dataset_split_part_file_paths("TACO")

# assert len(part_file_paths) == 26, "unexpected number of split parts?"
# assert len(dataset_paths) == len(part_file_paths), "unexpected number of dataset parts?"

all_problem_ids: list[str] = []
part_problem_ids: list[list[str]] = []
for part_file_path in part_file_paths:
    with part_file_path.open("r") as fd:
        target_problem_ids = yaml.safe_load(fd)
    part_problem_ids.append(target_problem_ids)
    assert not any([pid in all_problem_ids for pid in target_problem_ids]), "problem id already seen?"
    all_problem_ids.extend(target_problem_ids)

print(f"dataset metadata parsed: expecting {len(all_problem_ids)} problems over {len(dataset_paths)} parts")

In [ ]:
for dataset_path, expected_problem_ids in zip(dataset_paths, part_problem_ids):
    print(f"validating: {dataset_path}")
    try:
        reader = pyine.data.traces.dataset_reader.DatasetReader(dataset_path)
        found_problem_ids = [problem_id for problem_id in expected_problem_ids if problem_id in reader.problem_keys]
        print(f"\tfound problems ratio: {len(found_problem_ids) / len(expected_problem_ids):.2f}")
        unexpected_problem_ids = [
            problem_id for problem_id in reader.problem_keys if problem_id not in expected_problem_ids
        ]
        print(f"\tunexpected problems: {len(unexpected_problem_ids)}")
    except Exception as e:
        print(f"\tfailed: {e}")